# TASK 9 : Export the List of Inactive Customers

## Step 9.1 Setup the right context for your worksheet

**IMPORTANT:** Make sure to setup the right context for this notebook.
The current role determines which operations can be performed on Snowflake objects based on the access control privileges granted to the role.
First select a role, then select a warehouse that the role has privileges to use. Then select Database and the Schema.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE COURSERA;
USE SCHEMA PUBLIC;

## Step 9.2 Create a View to scale and save your work

Bring together all bits and pieces from previous tasks (Tasks 1-8):
- Extracting First & Last Names
- Extracting Dates
- Calculating new Columns
- Eliminating Nulls & Blanks
- Eliminating Duplicates

### Tasks 1-7: Clean and transform customer data

In [ ]:
%%sql -r dataframe_4
SELECT ID,
       SPLIT_PART(TRIM(NAME, ' 0'), ',', 1) AS FIRST_NAME,
       SPLIT_PART(TRIM(NAME, ' 0'), ', ', 2) AS LAST_NAME,
       EMAIL,
       TO_DATE(DOB, 'MMMM DD, YYYY') AS DOB,
       DATEDIFF('YEAR', TO_DATE(DOB, 'MMMM DD, YYYY'), CURRENT_DATE()) AS AGE,
       TO_DATE(LASTTRANSACTION, 'AUTO') AS LASTTRANSACTION,
       DATEDIFF('DAY', TO_DATE(LASTTRANSACTION, 'AUTO'), CURRENT_DATE()) AS DAYS_SINCE_LAST_TRANS,
       IFF(COMPANY IS NULL OR COMPANY = '', 'N/A', COMPANY) AS COMPANY,
       LTRIM(PHONE, '+0') AS PHONE,
       ADDRESS,
       POSTALZIP,
       REGION,
       COUNTRY
  FROM COURSERA.PUBLIC.CUSTOMERS
 WHERE NOT (EMAIL IS NULL OR EMAIL = '');

### Task 8: Check for duplicates

In [ ]:
%%sql -r dataframe_5
SELECT ID, NAME, EMAIL, LASTTRANSACTION,
       RANK() OVER (PARTITION BY EMAIL ORDER BY TO_DATE(LASTTRANSACTION, 'AUTO') DESC) AS RANK
  FROM COURSERA.PUBLIC.CUSTOMERS
QUALIFY RANK = 1;

### Create the CUSTOMERS_VW View

Use a `WITH` clause (CTE) to combine both queries — cleaning/transforming and deduplication — into a single view.

**Documentation:** [WITH clause](https://docs.snowflake.com/en/sql-reference/constructs/with.html)

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE VIEW COURSERA.PUBLIC.CUSTOMERS_VW AS
WITH DEDUPED AS (
    SELECT EMAIL,
           RANK() OVER (PARTITION BY EMAIL ORDER BY TO_DATE(LASTTRANSACTION, 'AUTO') DESC) AS RNK
      FROM COURSERA.PUBLIC.CUSTOMERS
)
SELECT C.ID,
       SPLIT_PART(TRIM(C.NAME, ' 0'), ',', 1) AS FIRST_NAME,
       SPLIT_PART(TRIM(C.NAME, ' 0'), ', ', 2) AS LAST_NAME,
       C.EMAIL,
       TO_DATE(C.DOB, 'MMMM DD, YYYY') AS DOB,
       DATEDIFF('YEAR', TO_DATE(C.DOB, 'MMMM DD, YYYY'), CURRENT_DATE()) AS AGE,
       TO_DATE(C.LASTTRANSACTION, 'AUTO') AS LASTTRANSACTION,
       DATEDIFF('DAY', TO_DATE(C.LASTTRANSACTION, 'AUTO'), CURRENT_DATE()) AS DAYS_SINCE_LAST_TRANS,
       IFF(C.COMPANY IS NULL OR C.COMPANY = '', 'N/A', C.COMPANY) AS COMPANY,
       LTRIM(C.PHONE, '+0') AS PHONE,
       C.ADDRESS,
       C.POSTALZIP,
       C.REGION,
       C.COUNTRY
  FROM COURSERA.PUBLIC.CUSTOMERS C
  JOIN DEDUPED D ON C.EMAIL = D.EMAIL AND D.RNK = 1
 WHERE NOT (C.EMAIL IS NULL OR C.EMAIL = '');

## Step 9.3 Query CUSTOMERS_VW to find the list of inactive Customers

In [ ]:
%%sql -r dataframe_7
SELECT *
  FROM COURSERA.PUBLIC.CUSTOMERS_VW
 WHERE DAYS_SINCE_LAST_TRANS > 90;

## Step 9.4 Export results to CSV

Use the **Download** button in the results panel to export the inactive customers list as a CSV file.